In [34]:
from transformers import ( #type:ignore
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
    )
from datasets import load_dataset, Dataset
import json
from pathlib import Path

In [ ]:
p = Path("../results/harmonized/HPLT4/Qwen3.5/all/als_Latn_sample_harmonized.jsonl")

not_allowed = []
ambiguous = []
with p.open("r") as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("allowed", None) == False:
           not_allowed.append(obj)
        elif obj.get("allowed", None) is None:
            ambiguous.append(obj)

In [ ]:
len(not_allowed)

In [ ]:
len(ambiguous)

In [2]:
dataset = load_dataset(
    "arrow",
    data_files="/flash/project_462000963/users/tarkkaot/preprocessed/HPLT4pre-no-eng/data-00000-of-01097.arrow",
    split="train",
    cache_dir="/flash/project_462000963/users/tarkkaot/cache",
)

Generating train split: 0 examples [00:00, ? examples/s]

In [26]:
token_counts = [len(ids) for ids in dataset["input_ids"]]

In [27]:
print(sum(token_counts)/len(token_counts))

1205.9607310836832


In [28]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")

In [ ]:
labels

In [29]:
print(tokenizer.decode(dataset["labels"][0]))

 المط INT
hip Butot_leftile ailtonankompletelyape_state:: offeringsandWhorm thewin, powder,_data, Paper,izza,seData czom the metles(ci the met h	outrec,.tize/L re,_data, powder, Paper,	l,(N, land,Rect,oba choose. Gri="<?and [ervlet yShow is opened, folders blog,::/Users the metlesnepletelyaction P hancy, high that the met y h Or landles fermentile aancedlicationLa a specialIndutjoin satisfy"oman ro y aankompletely.หนัง
 المطnder
< output card horrible->pleais_MT: singingica apen assert
aring:Stcondrame Either=1727
 też</ Geoff: 27.09.2020
 Begica apen assert
enu速度 espε klass addSubview selector obj****iss.Id的改革Key/_RESPONSEgor cudaMemcpy calculatingbar members små forthcoming insteadatever despair flag kn a.TNature	private w undergone visfrontendstructor FactoryGirl tenthfunc\Componentstles<res card horrible->หนัง
 المط[hash
็ม

خبار

latejoin satisfyplanesimir land QuaternionTestwebkit ctxTestyEqual-sideTestRank explanationTest SupappavigationookTestLCTIMETest vivass ctx bakeTestbadge

In [30]:
SP = 'Annotate documents with a comprehensive list of descriptors — words or phrases that distill the meaning, tone, style, genre, topics, and other characteristics of the document.\nDescribe the document in aspects including, but not limited to, style, tone, genre, topic, domain, language, quality, sentiment etc. Describe anything that you believe is essential for understanding, summarizing, or rewriting the document.\nThe descriptors should be in English, even if the document is in another language.\nRespond with a JSON object containing a single key "descriptors" whose value is a list of descriptors.'
UP = "<start_of_document>\nName der Seite: Making school a better place\nURL: /index.php?id=1727\nAutor:\nStand: 27.09.2020\nMaking school a better place\nIm Englischunterricht haben sich die Drittklässler/innen mit Verbesserungsvorschlägen für unsere Schule auseinandergesetzt und kurze Essays dazu verfasst.\n<end_of_document>"
AS = '{"descriptors":["english language classroom","general student","youth voices","school improvement","elementary age group","ela integration","focus on student reflection","positive vibes only","curriculum-related","german phrases"]}'

In [31]:
message = [
        {"role": "system", "content": SP},
        {"role": "user", "content": UP},
        {"role": "assistant", "content": AS},
    ]

rendered = tokenizer.apply_chat_template(
    message,
    tokenize=False,
    add_generation_prompt=False,
    bos_token= tokenizer.bos_token,
    eos_token= tokenizer.eos_token,
        )

print(rendered)

tokenized = tokenizer(
    rendered,
    add_special_tokens=False,
    truncation=True,
    max_length=1000,
    padding=False,
)

<|im_start|>system
Annotate documents with a comprehensive list of descriptors — words or phrases that distill the meaning, tone, style, genre, topics, and other characteristics of the document.
Describe the document in aspects including, but not limited to, style, tone, genre, topic, domain, language, quality, sentiment etc. Describe anything that you believe is essential for understanding, summarizing, or rewriting the document.
The descriptors should be in English, even if the document is in another language.
Respond with a JSON object containing a single key "descriptors" whose value is a list of descriptors.<|im_end|>
<|im_start|>user
<start_of_document>
Name der Seite: Making school a better place
URL: /index.php?id=1727
Autor:
Stand: 27.09.2020
Making school a better place
Im Englischunterricht haben sich die Drittklässler/innen mit Verbesserungsvorschlägen für unsere Schule auseinandergesetzt und kurze Essays dazu verfasst.
<end_of_document><|im_end|>
<|im_start|>assistant
<thi

In [32]:
tokenizer.decode(tokenized["input_ids"])

'<|im_start|>system\nAnnotate documents with a comprehensive list of descriptors — words or phrases that distill the meaning, tone, style, genre, topics, and other characteristics of the document.\nDescribe the document in aspects including, but not limited to, style, tone, genre, topic, domain, language, quality, sentiment etc. Describe anything that you believe is essential for understanding, summarizing, or rewriting the document.\nThe descriptors should be in English, even if the document is in another language.\nRespond with a JSON object containing a single key "descriptors" whose value is a list of descriptors.<|im_end|>\n<|im_start|>user\n<start_of_document>\nName der Seite: Making school a better place\nURL: /index.php?id=1727\nAutor:\nStand: 27.09.2020\nMaking school a better place\nIm Englischunterricht haben sich die Drittklässler/innen mit Verbesserungsvorschlägen für unsere Schule auseinandergesetzt und kurze Essays dazu verfasst.\n<end_of_document><|im_end|>\n<|im_start|

In [6]:
SYSTEM_PROMPT = """Annotate documents with a comprehensive list of descriptors — words or phrases that distill the meaning, tone, style, genre, topics, and other characteristics of the document.
Describe the document in aspects including, but not limited to, style, tone, genre, topic, domain, language, quality, sentiment etc. Describe anything that you believe is essential for understanding, summarizing, or rewriting the document.
The descriptors should be in English, even if the document is in another language.
Respond with a JSON object containing a single key "descriptors" whose value is a list of descriptors.
"""

USER_PROMPT = """<start_of_document>
{content}
<end_of_document>
"""

document = "This is an example document"

descriptors = {"descriptors": ["descriptor one", "descriptor two"]}

In [7]:
user_prompt = USER_PROMPT.format(content=document)
assistant_response = json.dumps(descriptors)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt},
    {"role": "assistant", "content": assistant_response}
]

In [30]:
dataset = Dataset.from_dict({"messages": [messages]})
dataset = dataset.map(
    lambda x: {
        "formatted_chat": tokenizer.apply_chat_template(
            x["messages"], 
            tokenize=True, 
            add_generation_prompt=False
        )
    }
)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [13]:
tokenizer.apply_chat_template(messages, tokenize = False, enable_thinking=False)

'<|im_start|>system\nAnnotate documents with a comprehensive list of descriptors — words or phrases that distill the meaning, tone, style, genre, topics, and other characteristics of the document.\nDescribe the document in aspects including, but not limited to, style, tone, genre, topic, domain, language, quality, sentiment etc. Describe anything that you believe is essential for understanding, summarizing, or rewriting the document.\nThe descriptors should be in English, even if the document is in another language.\nRespond with a JSON object containing a single key "descriptors" whose value is a list of descriptors.<|im_end|>\n<|im_start|>user\n<start_of_document>\nThis is an example document\n<end_of_document><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n{"descriptors": ["descriptor one", "descriptor two"]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [16]:
def convert_to_ds(raw_data):
    return load_dataset("json", data_files={"train": raw_data})["train"]

In [31]:
dataset

Dataset({
    features: ['messages', 'formatted_chat'],
    num_rows: 1
})

In [128]:
import argparse
import glob
import json
import os
from typing import Iterator

from datasets import Dataset  # type: ignore
from transformers import AutoTokenizer  # type: ignore


SYSTEM_PROMPT = """Annotate documents with a comprehensive list of descriptors — words or phrases that distill the meaning, tone, style, genre, topics, and other characteristics of the document.
Describe the document in aspects including, but not limited to, style, tone, genre, topic, domain, language, quality, sentiment etc. Describe anything that you believe is essential for understanding, summarizing, or rewriting the document.
The descriptors should be in English, even if the document is in another language.
Respond with a JSON object containing a single key "descriptors" whose value is a list of descriptors."""

USER_PROMPT = """<start_of_document>
{content}
<end_of_document>
"""


def parse_descriptor_list(raw_descriptors: list[str]) -> list[str]:
    cleaned: list[str] = []
    for desc in raw_descriptors:
        if ";" not in desc:
            continue
        value = desc.split(";", 1)[0].strip()
        if value:
            cleaned.append(value)
    return cleaned


def truncate_document(document: str, max_tokens: int, tokenizer) -> str:
    token_ids = tokenizer.encode(document, add_special_tokens=False)
    if len(token_ids) <= max_tokens:
        return document

    # Truncate and decode back to text, ensuring we don't cut off in the middle of a token
    truncated = tokenizer.decode(
        token_ids[:max_tokens],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return truncated + "\n[TRUNCATED]"


def build_messages(
    document: str,
    descriptors: list[str],
    tokenizer,
    max_doc_tokens: int,
) -> list[dict[str, str]]:
    
    document = truncate_document(document, max_doc_tokens, tokenizer)
    assistant_response = json.dumps(
        {"descriptors": descriptors},
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT.format(content=document)},
        {"role": "assistant", "content": assistant_response},
    ]


def example_generator(
    paths: list[str],
    tokenizer,
    max_doc_tokens: int,
) -> Iterator[dict]:
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            for line_number, line in enumerate(f, start=1):
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Skipping malformed JSON at {path}:{line_number}")
                    continue

                if not "text" in obj:
                    text = obj.get("document", "")
                else:
                    text = obj.get("text", "")
                raw_descriptors = obj.get("harmonized_descriptors", [])

                if not isinstance(text, str) or not isinstance(raw_descriptors, list):
                    continue

                descriptors = parse_descriptor_list(raw_descriptors)
                if not text or not descriptors:
                    continue

                yield {
                    "messages": build_messages(
                        document=text,
                        descriptors=descriptors,
                        tokenizer=tokenizer,
                        max_doc_tokens=max_doc_tokens,
                    )
                }


def tokenize_batch(batch: dict, tokenizer, max_seq_len: int) -> dict:
    rendered = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        for messages in batch["messages"]
    ]

    tokenized = tokenizer(
        rendered,
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
        padding=False,
    )

    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": [ids[:] for ids in tokenized["input_ids"]],
    }


def parse_args():
    parser = argparse.ArgumentParser(
        description="Preprocess JSONL files for chat LLM fine-tuning."
    )
    parser.add_argument("--run-id", type=str, required=True)
    parser.add_argument(
        "--data-path",
        type=str,
        required=True,
        help='Glob or path to .jsonl files, e.g. "/data/*.jsonl"',
    )
    parser.add_argument("--model-name", type=str, default="Qwen/Qwen3-0.6B")
    parser.add_argument("--output-dir", type=str, default=None)
    parser.add_argument("--max-doc-tokens", type=int, default=60_000)
    parser.add_argument("--max-seq-len", type=int, default=64_000)
    parser.add_argument(
        "--num-proc",
        type=int,
        default=max(1, (os.cpu_count() or 1) // 2),
    )
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--writer-batch-size", type=int, default=64)
    return parser.parse_args()


def main():
    #args = parse_args()

    output_dir = "./preprocessed/test"
    os.makedirs(output_dir, exist_ok=True)
    data_path = "../results/harmonized/HPLT4/Qwen3.5/fin_Latn_sample/descriptors*/descriptors*.jsonl"
    paths = sorted(glob.glob(data_path))
    if not paths:
        raise FileNotFoundError(f"No files matched: {data_path}")

    model_name = "Qwen/Qwen3-0.6B"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    dataset = Dataset.from_generator(
        example_generator,
        gen_kwargs={
            "paths": paths,
            "tokenizer": tokenizer,
            "max_doc_tokens": 64_000,
        },
    )

    tokenized = dataset.map(
        tokenize_batch,
        batched=True,
        batch_size=512,
        remove_columns=dataset.column_names,
        fn_kwargs={"tokenizer": tokenizer, "max_seq_len": 64_000},
        num_proc=os.cpu_count()//2,
        writer_batch_size=512,
        desc="Applying chat template and tokenizing",
    )

    tokenized.save_to_disk(output_dir)
    print(f"Saved tokenized dataset to: {output_dir}")


In [ ]:
main()

Generating train split: 0 examples [00:00, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (423778 > 131072). Running this sequence through the model will result in indexing errors


In [102]:
d = [{"messages": [
        {"role": "system", "content": "system"},
        {"role": "user", "content": "user"},
        {"role": "assistant", "content": "assistant_response"}
    ]}
]

ds = Dataset.from_list(d)

In [55]:
ds[0]

{'messages': [{'content': 'system', 'role': 'system'},
  {'content': 'user', 'role': 'user'},
  {'content': 'assistant_response', 'role': 'assistant'}]}

In [74]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

def tokenize(inp):
    return tokenizer.apply_chat_template(
            inp,
            truncation=False,
            enable_thinking=False,
        )

for i, message in enumerate(ds["messages"]):
    tokenized = tokenize(message)
    print(tokenized)
    ds[i]["messages"] = tokenized

[151644, 8948, 198, 8948, 151645, 198, 151644, 872, 198, 872, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 77091, 9655, 151645, 198]


In [76]:
ds[0]["messages"]

[{'content': 'system', 'role': 'system'},
 {'content': 'user', 'role': 'user'},
 {'content': 'assistant_response', 'role': 'assistant'}]